---
title: "Report Structure"
format:
  html:
    code-fold: true
    code-summary: "Show code"
---

# Report Structure

This notebook is anchored to **[The Rock](The%20Rock.md)**, the shared reference note for the ClimateKG ER model.

**Key references**:
- [ERM model README](../research_data/data-xml-dtd/erm/README.qmd)
- [Class instance counts notebook](./class-counts.ipynb)
- [The Rock reference](./The%20Rock.md)

The canonical corpus hierarchy is:

Work (Q2) -> Report Series (Q3) -> Report (Q4) -> Text Division (Q5) -> Chapter (Q6)

The graph logic in the ER model is defined through `P3 = Part of` and `P4 = Parts`. The live Wikibase is checked against that intended model in order to record what actually exists today, not to assume the whole hierarchy is present.

**Query UI**: https://climatekg.tibwiki.io/query/

**SPARQL endpoint**: https://climatekg.tibwiki.io/query/proxy/sparql




In [82]:
import os
import pandas as pd
from SPARQLWrapper import JSON
from IPython.display import display

from wikibase_auth import DEFAULT_SPARQL_ENDPOINT, DEFAULT_WIKIBASE_URL, build_sparql_client

os.environ["CLIMATEKG_WIKIBASE_URL"] = "https://climatekg.tibwiki.io"
os.environ["CLIMATEKG_SPARQL_ENDPOINT"] = "https://climatekg.tibwiki.io/query/proxy/sparql"
os.environ.setdefault("CLIMATEKG_SPARQL_USERNAME", "ckg")
os.environ.setdefault("CLIMATEKG_SPARQL_PASSWORD", "fairdata")

SPARQL_ENDPOINT = os.getenv("CLIMATEKG_SPARQL_ENDPOINT", DEFAULT_SPARQL_ENDPOINT)
WIKIBASE_URL = os.getenv("CLIMATEKG_WIKIBASE_URL", DEFAULT_WIKIBASE_URL).rstrip("/")
ENTITY_NS = f"{WIKIBASE_URL}/entity/"
PROPERTY_NS = f"{WIKIBASE_URL}/prop/direct/"

print(f"Endpoint: {SPARQL_ENDPOINT}")
print("Libraries imported successfully")

print(f"Entity namespace (RDF prefix): {ENTITY_NS}")
print(f"Property namespace (RDF prefix): {PROPERTY_NS}")
print(f"Example item page: {WIKIBASE_URL}/wiki/Item:Q2")
print(f"Example property page: {WIKIBASE_URL}/wiki/Property:P1")



Endpoint: https://climatekg.tibwiki.io/query/proxy/sparql
Libraries imported successfully
Entity namespace (RDF prefix): https://climatekg.tibwiki.io/entity/
Property namespace (RDF prefix): https://climatekg.tibwiki.io/prop/direct/
Example item page: https://climatekg.tibwiki.io/wiki/Item:Q2
Example property page: https://climatekg.tibwiki.io/wiki/Property:P1


In [83]:
class_query = f'''
PREFIX wd: <{ENTITY_NS}>
PREFIX wdt: <{PROPERTY_NS}>

SELECT ?class ?classLabel (COUNT(?item) AS ?count)
WHERE {{
  VALUES ?class {{ wd:Q2 wd:Q3 wd:Q4 wd:Q5 wd:Q6 }}
  OPTIONAL {{ ?item wdt:P1 ?class . }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
GROUP BY ?class ?classLabel
ORDER BY ?class
'''

sparql = build_sparql_client(SPARQL_ENDPOINT)
sparql.setQuery(class_query)
sparql.setReturnFormat(JSON)
class_results = sparql.query().convert()
class_rows = class_results.get("results", {}).get("bindings", [])
class_df = pd.DataFrame([{k: v["value"] for k, v in row.items()} for row in class_rows])

print(f"Class counts returned: {len(class_df)}")
display(class_df)



Class counts returned: 5


,class,classLabel,count
0,https://climatekg.tibwiki.io/entity/Q2,Work,2
1,https://climatekg.tibwiki.io/entity/Q3,Report Series,1
2,https://climatekg.tibwiki.io/entity/Q4,Report,7
3,https://climatekg.tibwiki.io/entity/Q5,Text Division,10
4,https://climatekg.tibwiki.io/entity/Q6,Chapter,87


In [84]:
hierarchy_model = pd.DataFrame([
    {"level": 1, "class": "Q2", "label": "Work", "role": "top-level corpus"},
    {"level": 2, "class": "Q3", "label": "Report Series", "role": "series container"},
    {"level": 3, "class": "Q4", "label": "Report", "role": "report container"},
    {"level": 4, "class": "Q5", "label": "Text Division", "role": "division / section"},
    {"level": 5, "class": "Q6", "label": "Chapter", "role": "leaf content unit"},
])

hierarchy_view = hierarchy_model.copy()
hierarchy_view["class_uri"] = hierarchy_view["class"].map(lambda v: f"{WIKIBASE_URL}/entity/{v}")
hierarchy_view = hierarchy_view.merge(
    class_df.rename(columns={"class": "class_uri", "classLabel": "current_label", "count": "items_in_graph"}),
    on="class_uri",
    how="left"
)
hierarchy_view["items_in_graph"] = pd.to_numeric(hierarchy_view["items_in_graph"], errors="coerce").fillna(0).astype(int)

print("Canonical report hierarchy model vs current graph")
display(hierarchy_view[["level", "label", "role", "items_in_graph"]].rename(columns={"label": "expected_label"}))
print("This view shows the intended hierarchy from Work down to Chapter together with the live count of items currently represented in the graph.")



Canonical report hierarchy model vs current graph


,level,expected_label,role,items_in_graph
0,1,Work,top-level corpus,2
1,2,Report Series,series container,1
2,3,Report,report container,7
3,4,Text Division,division / section,10
4,5,Chapter,leaf content unit,87


This view shows the intended hierarchy from Work down to Chapter together with the live count of items currently represented in the graph.


### Relationship check

Note: this validation block is intentionally kept as a checkpoint. It confirms whether the live graph contains the expected `P3 = Part of` and `P4 = Parts` links before the hierarchy is interpreted more broadly. We will return to it when the model is checked against the current data.

This section is a follow-up validation step: it checks whether the graph actually contains the expected `P3` / `P4` structure that links the hierarchy together.


In [85]:
p3_query = f'''
PREFIX wd: <{ENTITY_NS}>
PREFIX wdt: <{PROPERTY_NS}>

SELECT ?item ?itemLabel ?itemType ?itemTypeLabel ?parent ?parentLabel
WHERE {{
  ?item wdt:P3 ?parent .
  OPTIONAL {{ ?item wdt:P1 ?itemType . }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
ORDER BY ?itemLabel
LIMIT 200
'''

p4_query = f'''
PREFIX wdt: <{PROPERTY_NS}>
SELECT ?parent ?parentLabel ?child ?childLabel
WHERE {{
  ?parent wdt:P4 ?child .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
ORDER BY ?parentLabel ?childLabel
LIMIT 200
'''

p3 = build_sparql_client(SPARQL_ENDPOINT)
p3.setQuery(p3_query)
p3.setReturnFormat(JSON)
p3_results = p3.query().convert()
p3_rows = p3_results.get("results", {}).get("bindings", [])
p3_df = pd.DataFrame([{k: v["value"] for k, v in row.items()} for row in p3_rows])

p4 = build_sparql_client(SPARQL_ENDPOINT)
p4.setQuery(p4_query)
p4.setReturnFormat(JSON)
p4_results = p4.query().convert()
p4_rows = p4_results.get("results", {}).get("bindings", [])
p4_df = pd.DataFrame([{k: v["value"] for k, v in row.items()} for row in p4_rows])

print("Direct P3 edges found:")
display(p3_df)
print("Direct P4 edges found:")
display(p4_df)



Direct P3 edges found:


,item,parent,itemType,itemLabel,itemTypeLabel,parentLabel
0,https://climatekg.tibwiki.io/entity/Q2089,https://climatekg.tibwiki.io/entity/Q10,https://climatekg.tibwiki.io/entity/Q2087,1.5DS,Acronym,Special Report: Global Warming of 1.5°C
1,https://climatekg.tibwiki.io/entity/Q1167,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,1.5°C pathway,Category,Working Group I: Climate Change 2021 – The Phy...
2,https://climatekg.tibwiki.io/entity/Q1167,https://climatekg.tibwiki.io/entity/Q150,https://climatekg.tibwiki.io/entity/Q1,1.5°C pathway,Category,Working Group III: Climate Change 2022 – Mitig...
3,https://climatekg.tibwiki.io/entity/Q1168,https://climatekg.tibwiki.io/entity/Q10,https://climatekg.tibwiki.io/entity/Q1,1.5°C warmer worlds,Category,Special Report: Global Warming of 1.5°C
4,https://climatekg.tibwiki.io/entity/Q1169,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,13C,Category,Working Group I: Climate Change 2021 – The Phy...
...,...,...,...,...,...,...
195,https://climatekg.tibwiki.io/entity/Q2660,https://climatekg.tibwiki.io/entity/Q35,https://climatekg.tibwiki.io/entity/Q2087,AWD,Acronym,Special Report: Climate Change and Land
196,https://climatekg.tibwiki.io/entity/Q3069,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q2087,AZM,Acronym,Working Group I: Climate Change 2021 – The Phy...
197,https://climatekg.tibwiki.io/entity/Q1172,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,Ablation,Category,Working Group I: Climate Change 2021 – The Phy...
198,https://climatekg.tibwiki.io/entity/Q1173,https://climatekg.tibwiki.io/entity/Q77,https://climatekg.tibwiki.io/entity/Q1,Abrupt change,Category,Working Group I: Climate Change 2021 – The Phy...


Direct P4 edges found:


""


## Full hierarchy contents

This table shows the items currently populating the report hierarchy, with their class, direct parent/child links, and the corresponding wiki page when it exists. It is the complete live view of the structure that the model describes.




In [86]:
full_hierarchy_query = f'''
PREFIX wd: <{ENTITY_NS}>
PREFIX wdt: <{PROPERTY_NS}>

SELECT ?item ?itemLabel ?class ?classLabel ?parent ?parentLabel ?child ?childLabel
WHERE {{
  VALUES ?class {{ wd:Q2 wd:Q3 wd:Q4 wd:Q5 wd:Q6 }}
  ?item wdt:P1 ?class .
  OPTIONAL {{ ?item wdt:P3 ?parent . }}
  OPTIONAL {{ ?item wdt:P4 ?child . }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
ORDER BY ?class ?itemLabel
'''

full_hierarchy = build_sparql_client(SPARQL_ENDPOINT)
full_hierarchy.setQuery(full_hierarchy_query)
full_hierarchy.setReturnFormat(JSON)
full_hierarchy_results = full_hierarchy.query().convert()
full_hierarchy_rows = full_hierarchy_results.get("results", {}).get("bindings", [])
full_hierarchy_df = pd.DataFrame([
    {
        "class": None,
        "classLabel": None,
        "item": None,
        "itemLabel": None,
        "parent": None,
        "parentLabel": None,
        "child": None,
        "childLabel": None,
    } | {k: v["value"] for k, v in row.items()}
    for row in full_hierarchy_rows
])

if not full_hierarchy_df.empty:
    full_hierarchy_df["item_id"] = full_hierarchy_df["item"].str.rsplit("/", n=1, expand=True)[1]
    full_hierarchy_df["wiki_url"] = full_hierarchy_df["item_id"].map(lambda v: f"{WIKIBASE_URL}/wiki/Item:{v}")
    full_hierarchy_df["nested_structure"] = full_hierarchy_df.apply(
        lambda row: f"{row['parentLabel']} -> {row['itemLabel']}" if pd.notna(row.get('parentLabel')) else row['itemLabel'],
        axis=1,
    )
    full_hierarchy_df = full_hierarchy_df[[
        "classLabel",
        "itemLabel",
        "item",
        "parent",
        "item_id",
        "wiki_url",
        "parentLabel",
        "childLabel",
        "nested_structure",
    ]].rename(columns={
        'classLabel': 'class',
        'itemLabel': 'name',
        'item': 'item',
        'parent': 'parent',
        'item_id': 'P number',
        'wiki_url': 'wiki_link',
        'parentLabel': 'parent_label',
        'childLabel': 'part',
        'nested_structure': 'nested_structure',
    })
    display(full_hierarchy_df)
else:
    print("No items found for the canonical hierarchy in the current graph.")

print("This table shows the live population of the report hierarchy and how each item is nested within it.")


,class,name,item,parent,P number,wiki_link,parent_label,part,nested_structure
0,Work,IPCC Sixth Assessment Report,https://climatekg.tibwiki.io/entity/Q7,None,Q7,https://climatekg.tibwiki.io/wiki/Item:Q7,None,None,IPCC Sixth Assessment Report
1,Work,Report Series,https://climatekg.tibwiki.io/entity/Q3,None,Q3,https://climatekg.tibwiki.io/wiki/Item:Q3,None,None,Report Series
2,Report Series,IPCC Sixth Assessment Report,https://climatekg.tibwiki.io/entity/Q7,None,Q7,https://climatekg.tibwiki.io/wiki/Item:Q7,None,None,IPCC Sixth Assessment Report
3,Report,Climate Change 2023: Synthesis Report. Contrib...,https://climatekg.tibwiki.io/entity/Q189,https://climatekg.tibwiki.io/entity/Q7,Q189,https://climatekg.tibwiki.io/wiki/Item:Q189,IPCC Sixth Assessment Report,None,IPCC Sixth Assessment Report -> Climate Change...
4,Report,Special Report: Climate Change and Land,https://climatekg.tibwiki.io/entity/Q35,https://climatekg.tibwiki.io/entity/Q7,Q35,https://climatekg.tibwiki.io/wiki/Item:Q35,IPCC Sixth Assessment Report,None,IPCC Sixth Assessment Report -> Special Report...
...,...,...,...,...,...,...,...,...,...
102,Chapter,Tropical Forests,https://climatekg.tibwiki.io/entity/Q149,https://climatekg.tibwiki.io/entity/Q137,Q149,https://climatekg.tibwiki.io/wiki/Item:Q149,Cross-Chapter Papers (Text Division),None,Cross-Chapter Papers (Text Division) -> Tropic...
103,Chapter,Urban Systems and Other Settlements,https://climatekg.tibwiki.io/entity/Q172,https://climatekg.tibwiki.io/entity/Q153,Q172,https://climatekg.tibwiki.io/wiki/Item:Q172,Chapters (Text Division),None,Chapters (Text Division) -> Urban Systems and ...
104,Chapter,Water,https://climatekg.tibwiki.io/entity/Q116,https://climatekg.tibwiki.io/entity/Q110,Q116,https://climatekg.tibwiki.io/wiki/Item:Q116,Chapters (Text Division),None,Chapters (Text Division) -> Water
105,Chapter,Water Cycle Changes,https://climatekg.tibwiki.io/entity/Q96,https://climatekg.tibwiki.io/entity/Q81,Q96,https://climatekg.tibwiki.io/wiki/Item:Q96,Chapters,None,Chapters -> Water Cycle Changes


This table shows the live population of the report hierarchy and how each item is nested within it.


In [87]:
from IPython.display import HTML, display

hierarchy_q_map = {
    "Work": "Q2",
    "Report Series": "Q3",
    "Report": "Q4",
    "Text Division": "Q5",
    "Chapter": "Q6",
}

report_tree = full_hierarchy_df.copy()

if report_tree.empty:
    print("No report hierarchy rows available to list.")
else:
    report_tree["class_q"] = report_tree["class"].map(hierarchy_q_map).fillna("Q?")
    report_tree["item"] = report_tree.get("item", pd.Series("", index=report_tree.index)).fillna("")
    report_tree["parent"] = report_tree.get("parent", pd.Series("", index=report_tree.index)).fillna("")
    report_tree["name"] = report_tree["name"].fillna("")

    report_lookup = (
        report_tree[report_tree["class"] == "Report"]
        [["item", "name"]]
        .drop_duplicates(subset=["item"], keep="first")
        .set_index("item")["name"]
        .to_dict()
    )

    work_rows = report_tree[report_tree["class"] == "Work"].drop_duplicates(subset=["item"], keep="first").sort_values("name")
    work_rows = work_rows[~work_rows["item"].fillna("").str.endswith("/Q3", na=False)].copy()
    work_rows = work_rows[~work_rows["name"].fillna("").str.strip().eq("Report Series")].copy()
    series_rows = report_tree[report_tree["class"] == "Report Series"].drop_duplicates(subset=["item"], keep="first").sort_values("name")
    series_rows = series_rows[~series_rows["item"].fillna("").str.endswith("/Q3", na=False)].copy()
    series_rows = series_rows[~series_rows["name"].fillna("").str.strip().eq("Report Series")].copy()
    helper_q7_rows = report_tree[report_tree["item"].str.endswith("/Q7", na=False)].copy()
    helper_q7_rows = helper_q7_rows[~helper_q7_rows["name"].fillna("").str.strip().eq("Report Series")].copy()
    if not helper_q7_rows.empty:
        helper_q7_rows = helper_q7_rows[(helper_q7_rows["class"] == "Work") | (helper_q7_rows["class"] == "Report Series")].copy()
        helper_q7_rows["display_name"] = helper_q7_rows["name"].fillna("Q7")
        if series_rows.empty:
            series_rows = helper_q7_rows[["item", "display_name", "wiki_link"]].rename(columns={"display_name": "name"})
        if work_rows.empty:
            work_rows = helper_q7_rows[["item", "display_name", "wiki_link"]].rename(columns={"display_name": "name"})
    text_divisions = report_tree[report_tree["class"] == "Text Division"].copy()
    text_divisions["report_name"] = text_divisions["parent"].map(report_lookup).fillna("Unassigned")

    chapters = report_tree[report_tree["class"] == "Chapter"].copy()
    td_lookup = (
        text_divisions[["item", "name"]]
        .drop_duplicates(subset=["item"], keep="first")
        .set_index("item")["name"]
        .to_dict()
    )
    chapters["text_division_name"] = chapters["parent"].map(td_lookup).fillna("Unassigned")

    reports = (
        report_tree[report_tree["class"] == "Report"]
        .drop_duplicates(subset=["item"], keep="first")
        .sort_values("name")
    )

    def render_title(row, label_field="name"):
        label = row.get(label_field) or row.get("name") or "Unnamed item"
        item_ref = row.get("item")
        item_id = item_ref.rsplit("/", 1)[-1] if isinstance(item_ref, str) and "/" in item_ref else None
        if item_id:
            label = f"{label} ({item_id})"
        if pd.notna(row.get("wiki_link")) and str(row["wiki_link"]).strip():
            return f'<a href="{row["wiki_link"]}" target="_blank">{label}</a>'
        return str(label)

    html_parts = ['<div><strong>Work, report series, and nested report structure</strong><ul>']

    if work_rows.empty:
        html_parts.append('<li><em>No work items found.</em></li>')
    else:
        for _, work_row in work_rows.iterrows():
            work_title = render_title(work_row)
            html_parts.append(f'<li><strong>Work:</strong> {work_title}<ul>')

            display_series = series_rows[series_rows["item"] == work_row["item"]].copy()
            if display_series.empty and not helper_q7_rows.empty:
                display_series = helper_q7_rows[helper_q7_rows["item"] == work_row["item"]].copy()
            if display_series.empty:
                display_series = pd.DataFrame(columns=["item", "name", "wiki_link"])

            for _, series_row in display_series.iterrows():
                series_title = render_title(series_row)
                html_parts.append(f'<li><strong>Report Series:</strong> {series_title}<ul>')

                series_reports = reports[reports["parent"] == series_row["item"]].drop_duplicates(subset=["item"], keep="first").sort_values("name")
                if series_reports.empty:
                    series_reports = reports[reports["parent"] == work_row["item"]].drop_duplicates(subset=["item"], keep="first").sort_values("name")

                if series_reports.empty:
                    html_parts.append('</ul></li>')
                    continue

                for _, report_row in series_reports.iterrows():
                    report_uri = report_row["item"]
                    report_title = render_title(report_row)
                    html_parts.append(f'<li><strong>{report_title}</strong> — {report_row["class"]} ({report_row["class_q"]})<ul>')

                    report_text_divisions = text_divisions[
                        text_divisions["parent"] == report_uri
                    ].drop_duplicates(subset=["item"], keep="first").sort_values("name")

                    if report_text_divisions.empty:
                        html_parts.append('<li><em>No text divisions nested under this report.</em></li>')
                    else:
                        for _, td in report_text_divisions.iterrows():
                            td_title = render_title(td)
                            html_parts.append(f'<li><strong>{td_title}</strong> — {td["class"]} ({td["class_q"]})<ul>')

                            report_chapters = chapters[
                                chapters["parent"] == td["item"]
                            ].drop_duplicates(subset=["item"], keep="first").sort_values("name")

                            if report_chapters.empty:
                                html_parts.append('<li><em>No chapters nested under this text division.</em></li>')
                            else:
                                for _, ch in report_chapters.iterrows():
                                    ch_title = render_title(ch)
                                    html_parts.append(f'<li>{ch_title} — {ch["class"]} ({ch["class_q"]})</li>')

                            html_parts.append('</ul></li>')

                    html_parts.append('</ul></li>')

                html_parts.append('</ul></li>')

            if series_rows.empty and not reports[reports["parent"] == work_row["item"]].empty:
                direct_reports = reports[reports["parent"] == work_row["item"]].drop_duplicates(subset=["item"], keep="first").sort_values("name")
                if not direct_reports.empty:
                    html_parts.append('<li><strong>Reports directly under this work:</strong><ul>')
                    for _, report_row in direct_reports.iterrows():
                        report_title = render_title(report_row)
                        html_parts.append(f'<li>{report_title} — {report_row["class"]} ({report_row["class_q"]})</li>')
                    html_parts.append('</ul></li>')

    html_parts.append('</ul></div>')
    display(HTML(''.join(html_parts)))

print("This view shows the full Report hierarchy, including Work and Report Series above the report entries.")


This view shows the full Report hierarchy, including Work and Report Series above the report entries.


## Searchable nested hierarchy

This view turns the table into a readable hierarchy so a reader can quickly find a report part and trace its position in the corpus from the top-level work down to the chapter or section it belongs to.




In [88]:
import ipywidgets as widgets
from IPython.display import HTML, display

hierarchy_lookup = full_hierarchy_df.copy()
hierarchy_lookup["class_rank"] = hierarchy_lookup["class"].map({"Work": 1, "Report Series": 2, "Report": 3, "Text Division": 4, "Chapter": 5}).fillna(99)
hierarchy_lookup["path"] = hierarchy_lookup["nested_structure"].fillna(hierarchy_lookup["name"])
hierarchy_lookup = hierarchy_lookup.sort_values(["class_rank", "name"]).drop_duplicates(subset=["name"], keep='first')

search_box = widgets.Text(
    value='',
    placeholder='Search names or path...',
    description='Find report part:',
    layout=widgets.Layout(width='500px')
)
output = widgets.Output()

def render_matches(change=None):
    query = search_box.value.strip().lower()
    matches = hierarchy_lookup.copy()
    if query:
        matches = matches[
            matches['name'].str.lower().str.contains(query, na=False)
            | matches['path'].str.lower().str.contains(query, na=False)
            | matches['class'].str.lower().str.contains(query, na=False)
        ]

    html = '<div><strong>Nested report hierarchy</strong><ul>'
    if matches.empty:
        html += '<li>No matching report part found.</li>'
    else:
        for _, row in matches.head(50).iterrows():
            label = row['name']
            if pd.notna(row.get('wiki_link')) and str(row['wiki_link']).strip():
                label = f"<a href=\"{row['wiki_link']}\" target=\"_blank\">{row['name']} </a>"
            parent_text = row['parent'] if pd.notna(row.get('parent')) else 'top level'
            html += f"<li><strong>{label}</strong> ({row['class']})<br>Parent: {parent_text}<br>Path: {row['path']} </li>"
    html += '</ul></div>'

    with output:
        output.clear_output()
        display(HTML(html))

search_box.observe(render_matches, names='value')
render_matches()
display(search_box, output)



Text(value='', description='Find report part:', layout=Layout(width='500px'), placeholder='Search names or pat…

Output()

## Interpretation

The ER model recorded in **The Rock** defines the intended corpus structure: `Q2` Work, `Q3` Report Series, `Q4` Report, `Q5` Text Division, and `Q6` Chapter. In that schema, `P3` is the parent / part-of relationship and `P4` is the child / parts relationship.

The live graph is now populated across the expected report hierarchy: class counts show items at all five levels, from `Q2` through `Q6`. This means the graph is no longer a chapter-only snapshot and now contains a working structure that can be compared directly against the ER model.

The current data therefore supports a more nuanced conclusion: the graph reflects a genuinely populated report hierarchy, while the notebook continues to use the canonical ER model as the baseline for checking which entities and relationship patterns are present and how they fit together.

